# Pengumpulan data harga komoditas (SIMONIKA)

Mengunduh file bulanan dari Google Drive, memetakan tab harian per pasar, dan memvalidasi kalender.

**Keluaran** (di `data/raw/`): `xlsx_asli/*.xlsx`, `_manifest.csv`, `_inventaris_tab.csv`.

**Aturan tanggal:** tanggal sebuah tab = bulan dan tahun dari nama file + nomor tab.
Judul tanggal di dalam tab sering tidak diperbarui saat sheet disalin, jadi hanya dipakai sebagai penanda (`judul_cocok`).

In [1]:
%pip install -q google-api-python-client google-auth-oauthlib pandas openpyxl python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Konfigurasi dan autentikasi Google Drive
`.env` berisi `GDRIVE_FOLDER_ID=...`; `credentials.json` dari Google Cloud. Keduanya (dan `token.json`) masuk `.gitignore`.

In [2]:
import io, os, re
from collections import Counter
from pathlib import Path

import openpyxl
import pandas as pd
from dotenv import load_dotenv
from google.auth.exceptions import RefreshError
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

In [3]:
ROOT = Path("..")
load_dotenv(ROOT / ".env")
FOLDER_ID = os.environ["GDRIVE_FOLDER_ID"]
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
RAW = ROOT / "data" / "raw"
XLSX_DIR = RAW / "xlsx_asli"
XLSX_DIR.mkdir(parents=True, exist_ok=True)


def get_credentials():
    token = ROOT / "token.json"
    creds = Credentials.from_authorized_user_file(str(token), SCOPES) if token.exists() else None
    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
        except RefreshError:
            creds = None
    if not (creds and creds.valid):
        flow = InstalledAppFlow.from_client_secrets_file(str(ROOT / "credentials.json"), SCOPES)
        creds = flow.run_local_server(port=0)
    token.write_text(creds.to_json())
    return creds


service = build("drive", "v3", credentials=get_credentials(), cache_discovery=False)
print("Terhubung sebagai:", service.about().get(fields="user(emailAddress)").execute()["user"]["emailAddress"])

Terhubung sebagai: erisalfionita18@gmail.com


## 2. Daftar file dan unduhan
File diunduh ulang hanya bila belum ada atau `modifiedTime` di Drive berubah. Riwayat versi disimpan di `_manifest.csv`.

In [4]:
SHEET = "application/vnd.google-apps.spreadsheet"
FOLDER = "application/vnd.google-apps.folder"
XLSX = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
MANIFEST = RAW / "_manifest.csv"


def list_files(folder_id):
    out, token = [], None
    while True:
        r = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,name,mimeType,modifiedTime)",
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        for f in r["files"]:
            if f["mimeType"] == FOLDER:
                out += list_files(f["id"])
            elif f["mimeType"] in (SHEET, XLSX):
                out.append(f)
        token = r.get("nextPageToken")
        if not token:
            return out


def download(f):
    req = (service.files().export_media(fileId=f["id"], mimeType=XLSX) if f["mimeType"] == SHEET
           else service.files().get_media(fileId=f["id"], supportsAllDrives=True))
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, req)
    done = False
    while not done:
        _, done = dl.next_chunk()
    f["path"].write_bytes(buf.getvalue())


# daftar file + nama lokal yang aman untuk Windows
df = pd.DataFrame(list_files(FOLDER_ID)).sort_values("name", ignore_index=True)
df["local"] = (df["name"].str.replace(r"\.xlsx$", "", regex=True)
                         .str.replace(r'[<>:"/\\|?*]', "_", regex=True)
                         .str.strip().str.rstrip("."))
dup = df["local"].duplicated(keep=False)
df.loc[dup, "local"] += "_" + df.loc[dup, "id"].str[:6]
df["path"] = df["local"].map(lambda s: XLSX_DIR / f"{s}.xlsx")

# unduh yang baru atau berubah saja
old = pd.read_csv(MANIFEST).set_index("id")["modifiedTime"] if MANIFEST.exists() else pd.Series(dtype=str)
df["status"] = "SUDAH TERBARU"
for i, f in df.iterrows():
    if f["path"].exists() and old.get(f["id"]) == f["modifiedTime"]:
        continue
    try:
        download(f)
        df.loc[i, "status"] = "DIUNDUH"
    except Exception as e:
        df.loc[i, "status"] = f"GAGAL: {e}"

# manifest hanya untuk yang berhasil (yang gagal dicoba lagi saat dijalankan ulang)
ok = ~df["status"].str.startswith("GAGAL")
df.loc[ok, ["id", "name", "modifiedTime"]].to_csv(MANIFEST, index=False)

print(f"Total file di Drive: {len(df)}")
print(df["status"].str.split(":").str[0].value_counts().to_string())

Total file di Drive: 44
status
SUDAH TERBARU    44


## 3. Peta tab harian
Tab harian = tab yang namanya angka (`1`, `01`, ...). Tab lain (`REKAPITULASI`, `GRAFIK`, ...) diabaikan.
Judul tanggal dibaca dari 8 baris pertama tiap tab (`HARI / TANGGAL : ...`).

In [5]:
rows, lain = [], Counter()
for _, f in df.iterrows():
    wb = openpyxl.load_workbook(f["path"], read_only=True, data_only=True)
    for name in wb.sheetnames:
        if re.fullmatch(r"\s*\d{1,2}\s*", name):
            head = " ".join(str(c) for r in wb[name].iter_rows(max_row=8, values_only=True)
                            for c in r if c is not None)
            m = re.search(r"TANGGAL\s*:?\s*(.+?\d{4})", head, re.I)
            rows.append({"file": f["name"], "tab": name.strip(), "tanggal_judul": m.group(1) if m else None})
        else:
            lain[name.strip().upper()] += 1
    wb.close()

df_tab = pd.DataFrame(rows)
print("Tab harian :", len(df_tab), "| tanpa tanggal di judul:", df_tab["tanggal_judul"].isna().sum())
print("Tab non-harian (diabaikan):", dict(lain))

Tab harian : 1336 | tanpa tanggal di judul: 3
Tab non-harian (diabaikan): {'REKAPITULASI': 44, 'GRAFIK': 5, 'SHEET4': 1, 'REKAPITULASI BULANAN': 1}


## 4. Tanggal tab dan validasi kalender

In [6]:
BULAN = {b: i for i, b in enumerate(
    ["JANUARI", "FEBRUARI", "MARET", "APRIL", "MEI", "JUNI", "JULI",
     "AGUSTUS", "SEPTEMBER", "OKTOBER", "NOVEMBER", "DESEMBER"], 1)}

def tgl(s):
    """'Minggu / 01 Januari 2023' -> Timestamp; NaT bila tidak terbaca."""
    m = re.search(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", s) if isinstance(s, str) else None
    if m and m.group(2).upper() in BULAN:
        return pd.Timestamp(int(m.group(3)), BULAN[m.group(2).upper()], int(m.group(1)))
    return pd.NaT


def periode(nama_file):
    """'07.REKAP JULI 2024.xlsx' -> Period 2024-07 (bulan dari awalan nomor, tahun dari nama)."""
    th = re.search(r"\d{4}", nama_file).group()
    return pd.Period(f"{th}-{nama_file[:2]}", "M")


df_tab["no"] = df_tab["tab"].astype(int)
df_tab["periode"] = df_tab["file"].map(periode)
df_tab["tgl_judul"] = df_tab["tanggal_judul"].map(tgl)
# tanggal data = bulan/tahun dari nama file + nomor tab (NaT bila nomor melebihi hari dalam bulan itu)
df_tab["tanggal"] = [pd.Timestamp(p.year, p.month, n) if n <= p.days_in_month else pd.NaT
                     for p, n in zip(df_tab["periode"], df_tab["no"])]
df_tab["judul_cocok"] = df_tab["tanggal"] == df_tab["tgl_judul"]

print("Nomor tab di luar bulan :", df_tab["tanggal"].isna().sum(), "(dilewati di tahap berikutnya)")
print("Nomor tab kembar        :", df_tab.duplicated(["periode", "no"]).sum())
print("Judul tidak cocok       :", (~df_tab["judul_cocok"]).sum(), "tab")
print(df_tab[~df_tab["judul_cocok"]].groupby("file").size().to_string())

semua = pd.date_range(df_tab["periode"].min().start_time, df_tab["periode"].max().end_time.normalize())
hilang = semua.difference(df_tab["tanggal"].dropna())
print(f"\nHari kalender: {len(semua):,} | hari tanpa tab: {[d.strftime('%Y-%m-%d') for d in hilang]}")

Nomor tab di luar bulan : 1 (dilewati di tahap berikutnya)
Nomor tab kembar        : 0
Judul tidak cocok       : 60 tab
file
01. REKAP JANUARI 2024.xlsx             1
01. REKAPITULASI JANUARI 2026.xlsx      8
02. REKAP FEBRUARI 2024.xlsx            1
02. REKAPITULASI FEBRUARI 2026.xlsx     1
03. REKAP MARET 2023.xlsx               1
03. REKAPITULASI MARET 2026.xlsx        4
04. REKAP APRIL 2023.xlsx               1
04. REKAP APRIL 2024.xlsx               2
04. REKAPITULASI APRIL 2025.xlsx        1
04. REKAPITULASI APRIL 2026.xlsx        3
05. REKAPITULASI MEI 2026.xlsx          1
06. REKAPITULASI JUNI 2026.xlsx        30
07. REKAPITULASI JULI 2025.xlsx         1
08. REKAPITULASI AGUSTUS 2026.xlsx      1
12. REKAP DESEMBER 2023.xlsx            2
12. REKAP DESEMBER 2024.xlsx            1
12. REKAPITULASI DESEMBER 2025.xlsx     1

Hari kalender: 1,339 | hari tanpa tab: ['2023-04-22', '2023-04-23', '2023-06-29', '2025-03-31']


## 5. Simpan inventaris tab

In [7]:
df_tab["path"] = df_tab["file"].map(dict(zip(df["name"], df["path"].astype(str))))
df_tab[["file", "tab", "tanggal", "judul_cocok", "path"]].to_csv(RAW / "_inventaris_tab.csv", index=False)
print("Tersimpan:", RAW / "_inventaris_tab.csv", f"({len(df_tab):,} tab)")

Tersimpan: ..\data\raw\_inventaris_tab.csv (1,336 tab)
